# Sesión 4 · Ver los datos

**Antes de empezar:** Entorno de ejecución → Cambiar tipo de entorno de ejecución
→ **R**, y después Archivo → Guardar una copia en Drive.

Este cuaderno continúa el trabajo de clase. Cubre las gráficas del programa, con
una definición de cada una y el criterio para elegirla.

In [ ]:
library(tidyverse)

In [ ]:
url <- "https://docs.google.com/spreadsheets/d/e/2PACX-1vTTKC57eWZVIQ9lwhoR5nVqYM4kgi5zA9yifa-YStdfdJwNe7ATs0p-TUCUwvjfcWmHmvsZDEK8VX4I/pub?gid=1326008067&single=true&output=csv"

grupos <- read_csv(url) |>
  select(
    momento  = Timestamp,
    genero   = `Género`,
    carrera  = Carrera,
    edad     = `Edad, en años cumplidos`,
    estatura = `Estatura, en metros`,
    traslado = `Tiempo de traslado a la universidad, en minutos`,
    calzado  = `Número de calzado`
  ) |>
  mutate(momento = mdy_hms(momento))

glimpse(grupos)

## El color se define con un código hexadecimal

Un color se escribe como `#RRGGBB`: dos dígitos para rojo, dos para verde y dos
para azul, en base 16. `#3A6B6F` es el verde azulado de este curso.

Definirlos una vez al principio y reutilizarlos mantiene todas las gráficas del
documento con la misma identidad.

In [ ]:
# Paleta del curso
AZUL  <- "#3A6B6F"   # color principal
GRIS  <- "#CCCCCC"   # para lo que va en segundo plano
TINTA <- "#34425A"   # para el texto de las etiquetas
ROJO  <- "#A4503C"   # para destacar una sola categoría

# Un tema común para todas las gráficas de este cuaderno
theme_set(theme_minimal(base_size = 13))

# Una variable categórica

## Gráfica de puntos

::: Cada observación se dibuja como un punto, y los puntos que comparten valor se
apilan. La altura de la pila es la frecuencia de esa categoría. :::

**Cuándo usarla:** con pocos casos, cuando interesa que se vea que detrás de cada
unidad hay una observación. Con muchos casos deja de leerse y conviene la barra.

In [ ]:
ggplot(data = grupos, mapping = aes(x = carrera)) +
  geom_dotplot(binaxis = "x", stackdir = "up", dotsize = 0.5, fill = AZUL) +
  scale_y_continuous(NULL, breaks = NULL) +   # la escala vertical no aporta aquí
  labs(title = "Un punto por cada integrante del grupo", x = NULL)

### Los nombres se encinan

Esa celda corrió sin ningún error y la gráfica es ilegible: los nombres de las
carreras son largos y se traslapan en el eje.

**R no avisa cuando una gráfica queda mal.** Ejecutó lo que se le pidió y devolvió
una imagen; que las etiquetas se encimen no es un error de sintaxis. Revisar la
salida es parte del trabajo, y no lo hace el programa.

`str_wrap()` parte un texto en renglones de un ancho máximo. Aplicado a la
variable del eje, acomoda los nombres largos en varias líneas.

In [ ]:
ggplot(data = grupos, mapping = aes(x = str_wrap(carrera, 18))) +
  geom_dotplot(binaxis = "x", stackdir = "up", dotsize = 0.5, fill = AZUL) +
  scale_y_continuous(NULL, breaks = NULL) +
  labs(title = "Un punto por cada integrante del grupo", x = NULL)

## Gráfica de barras

::: Representa la frecuencia de cada categoría con la longitud de una barra. Todas
las barras arrancan del mismo punto, así que el ojo compara los extremos. :::

**Cuándo usarla:** para una variable categórica, siempre. Es la opción por
defecto y la más fácil de leer con precisión.

Se construye por capas. Primero la básica:

In [ ]:
ggplot(data = grupos, mapping = aes(x = carrera)) +
  geom_bar()

Los nombres se encinan. Con las barras en horizontal se resuelve, cambiando `x` por `y`:

In [ ]:
ggplot(data = grupos, mapping = aes(y = carrera)) +
  geom_bar(fill = AZUL) +               # color de relleno, en hexadecimal
  labs(x = "Estudiantes", y = NULL)

### Ordenar las categorías

ggplot ordena alfabéticamente, que casi nunca es el orden que interesa. Para
ordenar por frecuencia se convierte la variable en **factor**.

Un factor es una variable categórica con sus niveles declarados y en un orden
definido. `fct_infreq()` ordena por frecuencia; `fct_rev()` invierte.

In [ ]:
ggplot(data = grupos, mapping = aes(y = fct_rev(fct_infreq(carrera)))) +
  geom_bar(fill = AZUL) +
  labs(x = "Estudiantes", y = NULL)

### Etiquetar los valores

Cuando importa la cifra exacta, se escribe sobre la barra y se puede quitar el
eje, que ya no aporta.

In [ ]:
ggplot(data = grupos, mapping = aes(y = fct_rev(fct_infreq(carrera)))) +
  geom_bar(fill = AZUL) +
  geom_text(stat = "count", aes(label = after_stat(count)),
            hjust = 1.4,        # posición horizontal de la etiqueta
            color = "white",    # color del texto
            size = 4) +
  labs(x = NULL, y = NULL) +
  theme(axis.text.x = element_blank(),
        panel.grid  = element_blank())

### En porcentaje

Para mostrar porcentajes conviene calcularlos antes, en una tabla que se puede
imprimir y verificar, en vez de dejar que ggplot los calcule por dentro.

`count()` cuenta y `n / sum(n)` convierte el conteo en proporción.

In [ ]:
tabla <- grupos |>
  count(carrera) |>                    # frecuencia absoluta de cada carrera
  mutate(prop = n / sum(n))            # proporción sobre el total

tabla

In [ ]:
ggplot(data = tabla, mapping = aes(x = prop, y = fct_reorder(carrera, prop))) +
  geom_col(fill = AZUL) +
  geom_text(aes(label = scales::percent(prop, accuracy = 0.1)),
            hjust = 1.2,        # dentro de la barra; con -0.2 queda fuera
            color = "white",
            size = 4) +
  scale_x_continuous(labels = scales::percent) +
  labs(x = "Porcentaje del grupo", y = NULL)

### El color como herramienta, no como decoración

Si todas las barras son del mismo color, ninguna destaca. Si una sola es de color
y el resto son grises, el ojo va directo a esa. Es la recomendación central de
Knaflic: el color se reserva para lo que se quiere que se vea.

In [ ]:
mayor <- grupos |> count(carrera) |> slice_max(n, n = 1) |> pull(carrera)

grupos |>
  mutate(destacar = if_else(carrera == mayor, "si", "no")) |>
  ggplot(mapping = aes(y = fct_rev(fct_infreq(carrera)), fill = destacar)) +
  geom_bar() +
  scale_fill_manual(values = c("si" = ROJO, "no" = GRIS)) +
  labs(x = "Estudiantes", y = NULL) +
  guides(fill = "none")          # la leyenda no aporta: el color ya se explica solo

## Gráfica de pastel

::: Divide un círculo en rebanadas proporcionales a cada categoría. El lector
compara ángulos y áreas. :::

**Cuándo usarla:** casi nunca. Solo con dos o tres categorías que sean fracciones
simples, y cuando importe mostrar que las partes suman un todo.

Antes de correr la celda: **¿cuál categoría es la mayor y por cuánto?**

In [ ]:
grupos |>
  count(carrera) |>
  ggplot(mapping = aes(x = "", y = n, fill = carrera)) +
  geom_col(width = 1) +
  coord_polar("y") +             # convierte las barras apiladas en un círculo
  labs(x = NULL, y = NULL, fill = NULL) +
  theme_void()

### Un caso real de participación de mercado

Cuatro proveedores. Contesta antes de seguir: **¿cuál tiene la mayor
participación, y de qué tamaño?**

In [ ]:
proveedores <- tibble(
  proveedor = c("Proveedor A", "Proveedor B", "Proveedor C", "Proveedor D"),
  cuota     = c(34, 31, 9, 26)
)

ggplot(data = proveedores, mapping = aes(x = "", y = cuota, fill = proveedor)) +
  geom_col(width = 1) +
  coord_polar("y") +
  labs(x = NULL, y = NULL, fill = NULL) +
  theme_void()

Los mismos datos, como barra ordenada:

In [ ]:
proveedores |>
  mutate(proveedor = fct_reorder(proveedor, cuota)) |>
  ggplot(mapping = aes(x = cuota, y = proveedor)) +
  geom_col(fill = GRIS) +
  geom_text(aes(label = paste0(cuota, "%")), hjust = 1.3, color = TINTA) +
  labs(x = NULL, y = NULL, caption = "Total: 100%") +
  theme(axis.text.x = element_blank(), panel.grid = element_blank())

El Proveedor A tiene **34%** y el B tiene **31%**.

Knaflic presenta este caso con el pastel dibujado en **3D**. La perspectiva
inclina el círculo y las rebanadas de abajo se ven más grandes de lo que son: con
esa gráfica la mayoría señala al Proveedor B, que es el segundo.

El ojo no traduce bien área ni ángulo a cantidad. Con rebanadas parecidas es muy
difícil decir cuál es mayor; con rebanadas distintas se puede decir cuál, pero no
por cuánto. Las barras comparten una línea base, y por eso el ojo compara los
extremos con precisión.

**Nunca en 3D.** La dona tiene el mismo problema y uno peor: pide comparar
longitudes de arco.

Si vas a usar un pastel, la pregunta de Knaflic es **por qué**. Si tienes una
respuesta, adelante. Si no, va una barra.

Y no es cuestión de gusto: Cleveland y McGill lo midieron. Pidieron a personas
leer la misma cantidad codificada de distintas formas y compararon el error. La
posición sobre una escala común resultó la más exacta; el ángulo y el área
quedaron muy por debajo.

> Knaflic, C. N. (2015). *Storytelling with Data*. Wiley, cap. 2.
> Cleveland, W. S. y McGill, R. (1984). Graphical Perception. *JASA*, 79(387), 531-554.

# Una variable numérica

## Histograma

::: Parte el rango de la variable en intervalos del mismo ancho y dibuja una barra
con la cantidad de observaciones que cae en cada uno. :::

**Cuándo usarlo:** para ver la **forma** de una variable numérica: dónde se
concentra, si es simétrica, si tiene una o dos modas, si hay valores lejanos.

Las barras se tocan porque el eje es continuo: cada una cubre un intervalo, no
una categoría.

In [ ]:
ggplot(data = grupos, mapping = aes(x = estatura)) +
  geom_histogram(binwidth = 0.05, fill = AZUL, color = "white") +
  labs(title = "Estatura del grupo", x = "Estatura (m)", y = "Personas")

### El ancho del intervalo es una decisión

Corre las dos celdas y compara. Los mismos datos, dos anchos, dos formas
distintas. Ninguna de las dos es falsa: quien elige el ancho elige la historia.

In [ ]:
ggplot(data = grupos, mapping = aes(x = estatura)) +
  geom_histogram(binwidth = 0.02, fill = AZUL, color = "white") +
  labs(x = "Estatura (m) · binwidth = 0.02", y = NULL)

In [ ]:
ggplot(data = grupos, mapping = aes(x = estatura)) +
  geom_histogram(binwidth = 0.10, fill = AZUL, color = "white") +
  labs(x = "Estatura (m) · binwidth = 0.10", y = NULL)

## Polígono de frecuencias

::: Une con una línea los puntos medios de las barras del histograma. :::

**Cuándo usarlo:** para comparar dos o más distribuciones en la misma gráfica.
Dos histogramas encimados se tapan; dos polígonos se leen a la vez.

In [ ]:
ggplot(data = grupos, mapping = aes(x = estatura)) +
  geom_freqpoly(binwidth = 0.05, color = AZUL, linewidth = 1) +
  labs(x = "Estatura (m)", y = "Personas")

Con una segunda variable, la comparación entre grupos:

In [ ]:
ggplot(data = grupos, mapping = aes(x = estatura, color = genero)) +
  geom_freqpoly(binwidth = 0.05, linewidth = 1) +
  labs(x = "Estatura (m)", y = "Personas", color = NULL)

## Ojiva, o frecuencia acumulada

::: Para cada valor, muestra qué proporción de las observaciones es menor o igual
que él. La curva siempre sube y termina en 1. :::

**Cuándo usarla:** para responder preguntas de umbral. Qué porcentaje mide menos
de 1.70, o por debajo de qué estatura está la mitad del grupo.

In [ ]:
ggplot(data = grupos, mapping = aes(x = estatura)) +
  stat_ecdf(color = AZUL, linewidth = 1) +
  scale_y_continuous(labels = scales::percent) +
  labs(x = "Estatura (m)", y = "Porcentaje acumulado")

# Dos variables

## Diagrama de caja

::: La caja va del primer al tercer cuartil, la línea de en medio es la mediana y
los bigotes llegan hasta el valor más lejano que no se considere atípico. Los
puntos sueltos son las observaciones fuera de ese alcance. :::

**Cuándo usarlo:** para comparar una variable numérica entre categorías. Muestra
centro, dispersión y valores extremos de varios grupos en el mismo espacio.

In [ ]:
ggplot(data = grupos, mapping = aes(x = estatura, y = genero)) +
  geom_boxplot(fill = AZUL, alpha = 0.35) +
  labs(x = "Estatura (m)", y = NULL)

El diagrama de caja resume. Si además interesa ver cada observación, se
superponen los puntos.

In [ ]:
ggplot(data = grupos, mapping = aes(x = estatura, y = genero)) +
  geom_boxplot(fill = GRIS, alpha = 0.5, outlier.shape = NA) +
  geom_jitter(height = 0.15, color = AZUL, size = 2, alpha = 0.7) +
  labs(x = "Estatura (m)", y = NULL)

## Gráfica de dispersión

::: Cada observación es un punto, ubicado por sus valores en dos variables
numéricas. :::

**Cuándo usarla:** para ver si dos variables numéricas se mueven juntas, en qué
dirección y con cuánta fuerza.

In [ ]:
ggplot(data = grupos, mapping = aes(x = estatura, y = calzado)) +
  geom_point(aes(color = genero), size = 3) +
  labs(x = "Estatura (m)", y = "Calzado", color = NULL)

**Para discutir:** si la nube sube de izquierda a derecha, ¿qué se puede afirmar?
¿Que ser alto **causa** tener el pie grande?

Que dos variables se muevan juntas no dice cuál mueve a cuál, ni si una tercera
las mueve a las dos. Es el tema de Análisis de Datos II.

# Series de tiempo

## Gráfica de línea

::: Une con una línea observaciones consecutivas de la misma variable a lo largo
del tiempo. :::

**Cuándo usarla:** cuando el eje horizontal es el tiempo y el orden de los puntos
significa algo. Para categorías sin orden no aplica: unir con línea sugiere una
continuidad que no existe.

In [ ]:
grupos |>
  count(minuto = floor_date(momento, "minute")) |>
  ggplot(mapping = aes(x = minuto, y = n)) +
  geom_line(color = AZUL, linewidth = 1) +
  geom_point(color = AZUL, size = 2) +
  labs(x = NULL, y = "Respuestas")

### Esto todavía no es una serie de tiempo

Una **serie de tiempo** es una secuencia de observaciones de la misma variable,
ordenadas en el tiempo, donde cada valor depende de los anteriores y el conjunto
suele tener tendencia.

La gráfica de arriba cuenta respuestas por minuto de un evento que duró unos
minutos: tiene eje temporal, y no tiene ni tendencia ni dependencia entre
observaciones.

El precio de una acción, el tipo de cambio y la inflación mensual sí lo son.
Sobre ellos el promedio del **nivel** dice poco, porque la serie sube y baja: se
trabaja con la variación de un periodo al siguiente.

Este curso trabaja con observaciones independientes entre sí. Las series de
tiempo quedan fuera, y también de *Análisis de Datos II*.

# Qué gráfica va con qué variable

| Tienes | Gráfica | En R | Para qué |
|---|---|---|---|
| Una categórica | Puntos | `geom_dotplot()` | Ver cada observación |
| Una categórica | Barras | `geom_bar()` | Comparar frecuencias con precisión |
| Una categórica | Pastel | `coord_polar()` | Partes de un todo, con dos o tres categorías |
| Una numérica | Histograma | `geom_histogram()` | La forma de la distribución |
| Una numérica | Polígono | `geom_freqpoly()` | Comparar dos distribuciones |
| Una numérica | Ojiva | `stat_ecdf()` | Preguntas de umbral |
| Categórica y numérica | Caja | `geom_boxplot()` | Comparar centro y dispersión entre grupos |
| Dos numéricas | Dispersión | `geom_point()` | Si se mueven juntas |
| Numérica en el tiempo | Línea | `geom_line()` | Evolución |

Se lee de izquierda a derecha: primero se determina el tipo de variable, después
se elige la gráfica.

Para lo que no está aquí, [The R Graph Gallery](https://r-graph-gallery.com/)
tiene el catálogo con el código de cada una.

# Tres decisiones que mejoran cualquier gráfica

1. **Ordenar** las categorías por su valor, salvo que tengan orden propio como
   los meses o los niveles de escolaridad. `fct_infreq()` y `fct_reorder()`.
2. **Reservar el color.** Gris para el contexto y color para lo que quieres que
   se vea. Si todo es de color, nada destaca.
3. **Quitar lo que no aporta.** Cuadrícula, fondo, ejes redundantes y leyendas
   que repiten lo que ya dicen las etiquetas.

Y una regla sin excepciones: **nunca en 3D.**

# Tarea

1. Del [catálogo de datos](https://cjjmdata.github.io/analisis_datos_i/datos/catalogo.html),
   elige una fuente de tu carrera y clasifica sus variables: cuáles son
   categóricas y cuáles numéricas, con su subtipo.
2. Busca una gráfica publicada en un informe o reporte de tu área y responde por
   escrito:
   - ¿Qué tipo de variable representa, y la gráfica le corresponde?
   - ¿Qué decisión tomó quien la hizo que pudo haber sido otra?
   - ¿Qué queda fuera que haría falta para juzgar lo que afirma?
3. Reproduce tres gráficas de este cuaderno con los datos del grupo, aplicando
   las tres decisiones de la sección anterior, y escribe una línea sobre qué
   muestra cada una.

**Guarda tu copia.**